<div dir="rtl">
<h1>چسباندن خروجی Headها و شمارش جدول Attention</h1>
<p>درس 42 از 76 · ادغام سرها و هزینهٔ زمینهٔ بلند · <code dir="ltr">36-merge-heads</code></p>
<p><a target="_self" href="http://127.0.0.1:8000/part-06/chapter-01/36-merge-heads.html">📖 بازگشت به همین درس</a></p>
<p>خروجی واقعی Headها را ادغام کنید و Projection نهایی را با ماژول پروژه تطبیق دهید.</p><p>پیش‌نیاز: تقسیم Head و شکل weights @ V را بشناسید.</p>
<p>این دفتر نیمهٔ عملی درس است. مثال‌ها آمادهٔ اجرا هستند؛ دو Cell با برچسب TODO را خودتان کامل کنید. پیام INCOMPLETE یعنی هنوز چیزی ننوشته‌اید، نه اینکه پاسخ درست است. جواب مرجع در این دفتر پنهان نشده است.</p>
<p>از بالا به پایین اجرا کنید. پس از تغییر هر تابع، Cell آن و سپس Cell آزمون را دوباره اجرا کنید. برای بررسی نهایی، از منوی <code>Kernel → Restart Kernel and Run All Cells</code> استفاده کنید.</p>
</div>

In [ ]:
from pathlib import Path
import os
import sys

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "mini_gpt").is_dir() and (p / "book_src").is_dir()), None)
if project_root is None:
    raise RuntimeError("Extract the complete learning project; open this notebook inside it.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Python:", sys.executable)
print("Project:", project_root)

<div dir="rtl">
<h2>قبل از اجرا، پیش‌بینی کنید</h2>
<p>اگر attended شکل (2,3,5,4) دارد، خروجی ادغام چه شکلی است؟ چرا reshape مستقیم ممکن است ترتیب موقعیت‌ها را مخلوط کند؟</p>
</div>

<div dir="rtl"><p>پیش‌بینی من: …</p></div>

In [ ]:
import math
import torch
torch.set_num_threads(1)
torch.manual_seed(17)
from mini_gpt.config import ModelConfig
from mini_gpt.attention import CausalSelfAttention
attention = CausalSelfAttention(ModelConfig(12,8,12,3,1,0.)).eval()
x = torch.randn(2,5,12)
trace = {}
with torch.no_grad():
    output = attention(x,trace=trace)
attended = trace['weighted_values']
print('per-head output:',attended.shape)

<div dir="rtl">
<h2>این بار شما کد بنویسید</h2>
<p>تابع merge_heads(attended) را بنویسید: (B,H,T,D) به (B,T,H*D). هر زمان باید ویژگی‌های Headهای همان زمان را کنار هم داشته باشد.</p>
</div>

In [ ]:
def merge_heads(attended):
    # TODO
    return None

In [ ]:
def test_exercise():
    result = merge_heads(attended)
    if result is None: return False
    torch.testing.assert_close(attention.output(result),output)
    for B,H,T,D in ((1,2,3,4),(2,3,2,1)):
        a = torch.arange(B*H*T*D).reshape(B,H,T,D)
        merged = merge_heads(a)
        assert merged.shape == (B,T,H*D)
        for h in range(H):
            assert torch.equal(merged[:,:,h*D:(h+1)*D],a[:,h])
    return True

exercise_complete = test_exercise()
print("PASS" if exercise_complete else "INCOMPLETE: complete the TODO first")

<div dir="rtl">
<h2>فقط یک عامل را تغییر دهید</h2>
<p>فقط T را دو برابر کنید و B=2 و H=3 و float32 را ثابت بگذارید. فقط حافظهٔ یک جدول وزن را حساب می‌کنیم، نه کل حافظهٔ مدل.</p>
</div>

In [ ]:
for T in (16,32,64):
    elements = 2*3*T*T
    print('T, elements, KiB:',T,elements,elements*4/1024)

<div dir="rtl">
<h2>خرابی را پیدا کنید</h2>
<p>Contiguous نمی‌تواند محور معنایی اشتباه را اصلاح کند. تابع repair_merge(a) را اصلاح کنید؛ دادهٔ arange کمک می‌کند محل اشتباه را ببینید.</p>
</div>

In [ ]:
marker = torch.arange(24).reshape(1,2,3,4)
print('wrong first position:',marker.contiguous().view(1,3,8)[0,0])
print('two correct head pieces:',marker[0,0,0],marker[0,1,0])

<div dir="rtl">
<h2>اصلاح را خودتان بنویسید</h2>
<p>علت را توضیح دهید، سپس تابع زیر را کامل کنید. خطای عمدی بالا یک نمونهٔ آموزشی است؛ آزمون پایین باید اصلاح شما را بسنجد.</p>
</div>

In [ ]:
def repair_merge(a):
    # TODO
    return None

In [ ]:
def test_repair():
    result = repair_merge(marker)
    if result is None: return False
    assert torch.equal(result[0,0],torch.cat((marker[0,0,0],marker[0,1,0])))
    a = torch.arange(60).reshape(2,3,5,2)
    assert torch.equal(repair_merge(a)[:,:,2:4],a[:,1])
    return True

repair_complete = test_repair()
print("PASS" if repair_complete else "INCOMPLETE: complete the TODO first")

<div dir="rtl">
<h2>در Mini-GPT کجا به کار می‌آید؟</h2>
<p>مقایسه با attention.output روی همان وزن‌ها انجام شد؛ Dropout صفر است. این بخش انتهای CausalSelfAttention مشترک میان v4 و MiniGPT نهایی است.</p>
</div>

<div dir="rtl">
<h2>با زبان خودتان توضیح دهید</h2>
<p>چرا دوبرابرکردن Batch و دوبرابرکردن T اثر یکسانی بر حافظهٔ جدول وزن ندارند؟</p>
</div>
<div dir="rtl"><p>پیش‌بینی و مشاهدهٔ من: …</p><p>علت خرابی و اصلاح من: …</p></div>

<div dir="rtl"><p><a target="_self" href="http://127.0.0.1:8000/part-06/chapter-01/36-merge-heads.html">بازگشت به درس و ادامهٔ مسیر</a> · <a target="_self" href="http://127.0.0.1:8000/answers/36-merge-heads.html#lab-solution">فقط پس از تلاش: راه‌حل مرجع آزمایشگاه</a></p></div>